In [1]:
from drive_service.auth_service import load_creds
from drive_service.drive_client import get_drive_service

creds = load_creds()

drive = get_drive_service(creds)


In [2]:
from drive_service.logging_utils import get_logger


logger = get_logger()

In [3]:
import os
from scan_directory.cli import _get_root_name
from pathlib import Path
root = "1guqKp2FFRjl4OAgRN5y-7wkBeWQWDl1S"
root_prefix = _get_root_name(drive, root)

scan_output =  Path( root_prefix)

os.makedirs(scan_output, exist_ok=True)
root_prefix


'PAGLIAI'

In [4]:
from drive_service.drive_client import list_children


sub_folders = list_children(drive, root)
print( "Sotto Cartelle", len(sub_folders))
sub_folders

Sotto Cartelle 20


[{'id': '1cXxRFkCHuIaXIRDaRtb99hQYf9FDKBUY',
  'name': 'TARFANELLI VIRGINIA',
  'mimeType': 'application/vnd.google-apps.folder'},
 {'id': '1b00TfG52KymRExYjhoLdRw73XIMLhqhS',
  'name': 'ZUMPANO GIUSEPPE',
  'mimeType': 'application/vnd.google-apps.folder'},
 {'id': '1eF7dhIHvsUxEzUznGNJQI6lR2uT-6DWV',
  'name': 'VIGGIANI MARIA ALESSIA',
  'mimeType': 'application/vnd.google-apps.folder'},
 {'id': '1hqPg5oAMBSW3BWoZynu3f0nPn2eNBLef',
  'name': 'SCHETTINO OLIMPIA',
  'mimeType': 'application/vnd.google-apps.folder'},
 {'id': '12ND3WhvsYtfzW6wJzL__JwnISBQ-fP8N',
  'name': 'TRAPPOLINI CHIARA',
  'mimeType': 'application/vnd.google-apps.folder'},
 {'id': '1OMYI9S5Xhu_IduiGZ6eLq0RtE2xCpRlG',
  'name': 'TATAVITTO FRANCA',
  'mimeType': 'application/vnd.google-apps.folder'},
 {'id': '1yrmCul5OZlL_VzysF2Pg_5E1fIKf59_k',
  'name': 'SPADINI ANGIOLO',
  'mimeType': 'application/vnd.google-apps.folder'},
 {'id': '1a9fKeXi-bNfCEytKHWdsU2GV-eLqlyAK',
  'name': 'STORRI LUCIA',
  'mimeType': 'applicat

In [5]:
from concurrent.futures import ThreadPoolExecutor, as_completed

from scan_directory.config import exclude_terms_normalized
from scan_directory.scan_service import build_folder_report

workers= 8 

reports = []
total_included = 0
with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = [
            pool.submit(
                build_folder_report,
                creds,
                emp,
                exclude_terms_normalized,
                root_prefix=root_prefix,
            )
            for emp in sub_folders
        ]
        for i, f in enumerate(as_completed(futures), 1):
            report = f.result()
            reports.append(report)
            total_included += report["counts"]["included"]
            logger.info(
                "Progress %s/%s employees, %s files",
                i,
                len(futures),
                total_included,
            )

Found file with excluded term buste paghe.pdf buste
Found file with excluded term BUSTE PAGA.pdf buste
Found file with excluded term Cedolino-2023-1.pdf cedolino
Found file with excluded term Cedolino-2023-10-1.pdf cedolino
Found file with excluded term Cedolino-2023-3.pdf cedolino
Found file with excluded term Cedolino-2023-11.pdf cedolino
Found file with excluded term Cedolino-2023-2.pdf cedolino
Found file with excluded term Cedolino-2023-8.pdf cedolino
Found file with excluded term Cedolino-2023-5.pdf cedolino
Found file with excluded term Cedolino-2023-12-1.pdf cedolino
Found file with excluded term Cedolino-2023-3-1.pdf cedolino
Found file with excluded term Cedolino-2023-4.pdf cedolino
Found file with excluded term Cedolino-2023-7.pdf cedolino
Found file with excluded term Cedolino-2023-9-1.pdf cedolino
Found file with excluded term Cedolino-2023-9.pdf cedolino
Found file with excluded term Cedolino-2023-6.pdf cedolino
Found file with excluded term Cedolino-2023-10.pdf cedolino


In [6]:
filtered = [file  for r in reports for file in r["filtered"]]
filtered

[{'employee': 'STORRI LUCIA',
  'employee_id': '1a9fKeXi-bNfCEytKHWdsU2GV-eLqlyAK',
  'file_id': '1nWa9l5EXuM0R24PHCnO5XoDB1gbBugb7',
  'file_name': 'BUSTE PAGA',
  'drive_path': '/PAGLIAI/STORRI LUCIA/BUSTE PAGA',
  'type': 'folder',
  'reason': 'buste paga'},
 {'employee': 'SCHETTINO OLIMPIA',
  'employee_id': '1hqPg5oAMBSW3BWoZynu3f0nPn2eNBLef',
  'file_id': '13ntu_SGGY--eazbGTP1DesTmFcWpu35j',
  'file_name': 'BUSTE PAGA',
  'drive_path': '/PAGLIAI/SCHETTINO OLIMPIA/BUSTE PAGA',
  'type': 'folder',
  'reason': 'buste paga'},
 {'employee': 'TRAPPOLINI CHIARA',
  'employee_id': '12ND3WhvsYtfzW6wJzL__JwnISBQ-fP8N',
  'file_id': '1pkp8jInV21sg9GM69hUlcRFOOQ_6CcIe',
  'file_name': 'buste paghe.pdf',
  'drive_path': '/PAGLIAI/TRAPPOLINI CHIARA/buste paghe.pdf',
  'type': 'file',
  'reason': 'buste'},
 {'employee': 'TRAPPOLINI CHIARA',
  'employee_id': '12ND3WhvsYtfzW6wJzL__JwnISBQ-fP8N',
  'file_id': '1I2DfmUbUfC1zZdf1cSPq160-Och6Mj2s',
  'file_name': 'BUSTE PAGA',
  'drive_path': '/PAGLI

In [7]:
from drive_service.schema import IndexFile


included_map: dict[str, IndexFile] = {}
filtered_map: dict[str, IndexFile] = {}

for report in reports:
    for item in report["included"]:
        file_id = item.get("file_id")
        if not file_id:
            continue
        if file_id in included_map:
            logger.warning("Duplicate file_id in included map: %s (last one wins)", file_id)
        included_map[file_id] = IndexFile(**item)
    for item in report["filtered"]:
        file_id = item.get("file_id")
        if not file_id:
            continue
        if file_id in filtered_map:
            logger.warning("Duplicate file_id in filtered map: %s (last one wins)", file_id)
        filtered_map[file_id] = IndexFile(**item)

In [8]:
from drive_service.index.map_index import MapIndex

included_path =  scan_output / "included_index.json"
filtered_path =  scan_output / "filtered_index.json" 
included_index = MapIndex.generate_index(root, len(sub_folders), included_map)
filtered_index = MapIndex.generate_index(root, len(sub_folders), filtered_map)
included_index.save_index(included_path)
filtered_index.save_index(filtered_path)